In [7]:
import os
import time
import math
import asyncio
import nest_asyncio
import mesa
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Patch Jupyter Concurrency
nest_asyncio.apply()
load_dotenv(override=True)
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# 2. Global Economics
PROMPT_COST = 0.075 / 1_000_000
RESPONSE_COST = 0.30 / 1_000_000

# 3. The Hardware Tool
def scan_sector(x: int, y: int) -> str:
    """Scans the physical grid sector for titanium resources."""
    print(f"   ⚙️ [HARDWARE] Drone scanning physical sector ({x}, {y})...")
    if x == 2 and y == 2: 
        return "MASSIVE TITANIUM DEPOSIT DETECTED."
    return "Barren rock. No resources."

# 4. The Vector Math
def cosine_similarity(v1, v2):
    dot = sum(a*b for a,b in zip(v1,v2))
    mag1 = math.sqrt(sum(a*a for a in v1))
    mag2 = math.sqrt(sum(b*b for b in v2))
    return dot / (mag1*mag2) if mag1 and mag2 else 0.0

class UnifiedAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.memory_db = []
    
    async def embed_text(self, text):
        res = await client.aio.models.embed_content(model='gemini-embedding-001', contents=text)
        return res.embeddings[0].values

    async def step_async(self):
        print(f"\n🚀 [AGENT {self.unique_id}] Booting up at Grid Coordinates {self.pos}...")
        
        # --- PHASE 9: RAG MEMORY ---
        query_vec = await self.embed_text("Where are the highest value resources located?")
        context = "No prior memories."
        if self.memory_db:
            scored = [(cosine_similarity(query_vec, m['vec']), m['text']) for m in self.memory_db]
            scored.sort(reverse=True)
            context = scored[0][1] # Extract the most relevant memory
        
        # --- PHASE 7: TOOL CONFIGURATION ---
        config = types.GenerateContentConfig(tools=[scan_sector], temperature=0.0)
        chat = client.aio.chats.create(model='gemini-2.5-flash', config=config)
        
        prompt = f"""
        You are an autonomous mining drone physically located at grid coordinates {self.pos}.
        Your most relevant past memory: {context}
        
        MISSION: 
        1. Call the 'scan_sector' tool using your EXACT current X and Y coordinates.
        2. Based on the scanner result, output a final status report.
        """
        
        try:
            # --- PHASE 8: ASYNC EXECUTION ---
            response = await chat.send_message(prompt)
            
            # --- PHASE 6: TOKEN ECONOMICS ---
            usage = getattr(response, 'usage_metadata', None)
            in_tok = usage.prompt_token_count if usage else 120
            out_tok = usage.candidates_token_count if usage else 30
            cost = (in_tok * PROMPT_COST) + (out_tok * RESPONSE_COST)
            self.model.log_economics(cost)
            
            print(f"   🤖 [DECISION] {response.text.strip()}")
            
            # Store the new memory dynamically
            new_mem = f"At coordinates {self.pos}, I found: {response.text.strip()}"
            vec = await self.embed_text(new_mem)
            self.memory_db.append({'text': new_mem, 'vec': vec})

        except Exception as e:
            print(f"   ❌ [ERROR] {e}")

class UnifiedModel(mesa.Model):
    def __init__(self):
        super().__init__()
        # --- PHASE 3: SPATIAL GRID ---
        self.grid = mesa.space.MultiGrid(5, 5, True)
        self.total_burn = 0.0
        
        # Place Agent 1 in the barren zone, and Agent 2 right on top of the Titanium
        a1 = UnifiedAgent(self)
        a2 = UnifiedAgent(self)
        self.grid.place_agent(a1, (0, 0)) 
        self.grid.place_agent(a2, (2, 2)) 

    def log_economics(self, cost):
        self.total_burn += cost

    async def run_step(self):
        # Fire all agents concurrently
        tasks = [a.step_async() for a in self.agents]
        await asyncio.gather(*tasks)

    def step(self):
        print("\n" + "="*60)
        print(f"⚡ INITIATING UNIFIED NEURO-SYMBOLIC STEP {self.steps + 1}")
        print("="*60)
        
        start = time.time()
        asyncio.run(self.run_step())
        end = time.time()
        
        print("-" * 60)
        print(f"⏱️ BATCH LATENCY: {end-start:.2f}s | 💰 TOTAL API BURN: ${self.total_burn:.6f}")
        print("="*60)
        self.steps += 1

In [8]:
import time

master_model = UnifiedModel()
for i in range(2):
    master_model.step()
    if i == 0:
        print("\n⏳ Throttle Engaged: Sleeping for 60 seconds to reset Google Free Tier API limits...")
        time.sleep(60)


⚡ INITIATING UNIFIED NEURO-SYMBOLIC STEP 2

🚀 [AGENT 1] Booting up at Grid Coordinates (0, 0)...

🚀 [AGENT 2] Booting up at Grid Coordinates (2, 2)...
   ⚙️ [HARDWARE] Drone scanning physical sector (2, 2)...
   ⚙️ [HARDWARE] Drone scanning physical sector (0, 0)...
   🤖 [DECISION] STATUS REPORT:
Scan of sector (2, 2) revealed a MASSIVE TITANIUM DEPOSIT. Mission successful.
   🤖 [DECISION] STATUS REPORT:
Sector (0, 0) scan complete. Result: Barren rock. No resources found.
------------------------------------------------------------
⏱️ BATCH LATENCY: 4.65s | 💰 TOTAL API BURN: $0.000040

⏳ Throttle Engaged: Sleeping for 60 seconds to reset Google Free Tier API limits...

⚡ INITIATING UNIFIED NEURO-SYMBOLIC STEP 4

🚀 [AGENT 1] Booting up at Grid Coordinates (0, 0)...

🚀 [AGENT 2] Booting up at Grid Coordinates (2, 2)...
   ⚙️ [HARDWARE] Drone scanning physical sector (2, 2)...
   ⚙️ [HARDWARE] Drone scanning physical sector (0, 0)...
   🤖 [DECISION] STATUS REPORT:
Sector (0, 0) scan com